# RAG pipeline — Qdrant only (no Superlinked)

Same data and text field as `02-RAG-preprocessing.ipynb`, but **only** OpenAI embeddings + `qdrant_client` (no Superlinked).

- **Embeddings:** OpenAI `text-embedding-3-small` (1536 dims by default).
- **Text embedded:** comma-joined categories (`categories_text`, same idea as 02).
- **Collection:** `yelp-businesses-qdrant-standalone-openai` — separate from Superlinked’s collection.
- **API key:** set `OPENAI_API_KEY` in `.env` at repo root (or run notebook cwd where `.env` lives), or rely on `api.core.config` when imported from project root.
- **Qdrant:** `docker compose up -d` (`http://localhost:6333`).


In [20]:
from __future__ import annotations

import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    FieldCondition,
    Filter,
    MatchAny,
    MatchValue,
    PointStruct,
    Range,
    VectorParams,
)

# Load repo-root .env when running from notebooks/
load_dotenv(Path("..") / ".env")

try:
    from api.core.config import config

    _key = (config.openai_api_key or "").strip()
except Exception:
    _key = ""

if not _key:
    _key = (os.getenv("OPENAI_API_KEY") or "").strip()

if not _key:
    raise ValueError("Set OPENAI_API_KEY in ../.env (repo root) or environment.")

openai_client = OpenAI(api_key=_key)

DATA_PATH = Path("../data/raw/yelp_academic_dataset_business_restaurants_with_hours_sample_1000.json")
QDRANT_URL = "http://localhost:6333"
COLLECTION_STANDALONE = "yelp-businesses-qdrant-standalone-openai"
EMBED_MODEL = "text-embedding-3-small"
VECTOR_NAME = "default"
EMBED_BATCH = 256  # inputs per embeddings.create call (stay under API limits)


In [21]:
def parse_categories_list(v) -> list[str]:
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return []
    if isinstance(v, list):
        return [str(x).strip() for x in v if str(x).strip()]
    s = str(v).strip()
    if not s:
        return []
    return [c.strip() for c in s.split(",") if c.strip()]


def categories_text_from_list(cats: list[str]) -> str:
    return ", ".join(cats)


def _as_dict(v):
    """Best-effort parser for nested object cells (dict or stringified dict)."""
    if isinstance(v, dict):
        return v
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return {}
    if isinstance(v, str):
        s = v.strip()
        if not s:
            return {}
        try:
            parsed = json.loads(s)
            return parsed if isinstance(parsed, dict) else {}
        except Exception:
            pass
        try:
            import ast
            parsed = ast.literal_eval(s)
            return parsed if isinstance(parsed, dict) else {}
        except Exception:
            return {}
    return {}


def _normalize_value(v):
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, (np.floating, float)):
        return float(v)
    if isinstance(v, (bool, np.bool_)):
        return bool(v)
    if isinstance(v, list):
        return ", ".join(map(str, v))
    return v


def flatten_df_businesses(df: pd.DataFrame) -> pd.DataFrame:
    """Flatten nested Yelp objects into top-level columns for Qdrant payload filtering."""
    out = df.copy()
    out["categories_text"] = out["categories"].map(lambda c: categories_text_from_list(parse_categories_list(c)))

    # attributes.*
    attrs = out["attributes"].map(_as_dict).apply(pd.Series)
    attrs.columns = [f"attributes_{c}" for c in attrs.columns]

    # hours.*
    hours = out["hours"].map(_as_dict).apply(pd.Series)
    hours.columns = [f"hours_{c}" for c in hours.columns]

    # Optional deeper flatten for BusinessParking under attributes
    if "attributes_BusinessParking" in attrs.columns:
        parking = attrs["attributes_BusinessParking"].map(_as_dict).apply(pd.Series)
        parking.columns = [f"attributes_BusinessParking_{c}" for c in parking.columns]
        attrs = pd.concat([attrs, parking], axis=1)

    flat = pd.concat([out.drop(columns=["attributes", "hours"], errors="ignore"), attrs, hours], axis=1)

    # Normalize types and make payload-safe values
    for c in flat.columns:
        flat[c] = flat[c].map(_normalize_value)

    return flat


def row_payload(row: pd.Series) -> dict:
    """Qdrant payload from a flattened row (non-null scalar values only)."""
    out = {}
    for k, v in row.to_dict().items():
        if v is None or (isinstance(v, float) and pd.isna(v)):
            continue
        if isinstance(v, (dict, list)):
            out[k] = json.dumps(v, default=str)
        elif isinstance(v, (np.integer,)):
            out[k] = int(v)
        elif isinstance(v, (np.floating, float)):
            out[k] = float(v)
        elif isinstance(v, (bool, np.bool_)):
            out[k] = bool(v)
        else:
            out[k] = str(v)
    return out


def embed_texts_openai(texts: list[str], *, model: str) -> np.ndarray:
    """Batch OpenAI embeddings; return (n, dim) float32 array."""
    all_rows: list[list[float]] = []
    for i in range(0, len(texts), EMBED_BATCH):
        chunk = texts[i : i + EMBED_BATCH]
        resp = openai_client.embeddings.create(model=model, input=chunk)
        chunk_embs = sorted(resp.data, key=lambda d: d.index)
        all_rows.extend([d.embedding for d in chunk_embs])
    return np.asarray(all_rows, dtype=np.float32)


In [22]:
df_businesses = pd.read_json(DATA_PATH, lines=True)
df_businesses_flat = flatten_df_businesses(df_businesses)

print("raw cols:", len(df_businesses.columns), "flat cols:", len(df_businesses_flat.columns))
[df_businesses_flat.columns[:20].tolist(), df_businesses_flat.head(2)]


raw cols: 14 flat cols: 63


[['business_id',
  'name',
  'address',
  'city',
  'state',
  'postal_code',
  'latitude',
  'longitude',
  'stars',
  'review_count',
  'is_open',
  'categories',
  'categories_text',
  'attributes_BikeParking',
  'attributes_RestaurantsPriceRange2',
  'attributes_RestaurantsDelivery',
  'attributes_BusinessAcceptsCreditCards',
  'attributes_RestaurantsTakeOut',
  'attributes_RestaurantsGoodForGroups',
  'attributes_GoodForKids'],
               business_id                       name           address  \
 0  cHZ1FUswyko119EkV_9aNQ          Upper Crust Pizza   1909 E Grant Rd   
 1  goTAL_3Ns-bsMgfGmKDq0A  Spitale's Deli & Catering  3309 Division St   
 
        city state postal_code   latitude   longitude  stars  review_count  \
 0    Tucson    AZ       85719  32.250429 -110.943524    3.5           180   
 1  Metairie    LA       70002  30.004793  -90.164386    4.5            53   
 
    ...  attributes_BusinessParking_validated attributes_BusinessParking_lot  \
 0  ...             

In [23]:
texts = df_businesses_flat["categories_text"].fillna("").astype(str).tolist()
embeddings = embed_texts_openai(texts, model=EMBED_MODEL)
emb_dim = int(embeddings.shape[1])
print("embedding dim:", emb_dim, "shape:", embeddings.shape)
assert embeddings.shape[0] == len(df_businesses_flat)


embedding dim: 1536 shape: (1000, 1536)


In [24]:
client = QdrantClient(url=QDRANT_URL, api_key="")

if client.collection_exists(COLLECTION_STANDALONE):
    client.delete_collection(COLLECTION_STANDALONE)

client.create_collection(
    collection_name=COLLECTION_STANDALONE,
    vectors_config={VECTOR_NAME: VectorParams(size=emb_dim, distance=Distance.COSINE)},
)

points = []
for pos in range(len(df_businesses_flat)):
    row = df_businesses_flat.iloc[pos]
    vec = embeddings[pos].tolist()
    payload = row_payload(row)
    payload["business_id"] = str(row["business_id"])  # preserve original Yelp id
    points.append(
        PointStruct(id=pos + 1, vector={VECTOR_NAME: vec}, payload=payload)
    )

client.upload_points(COLLECTION_STANDALONE, points)
print("uploaded", len(points), "points to", COLLECTION_STANDALONE)


/var/folders/rt/bmzmdtl92k7bz0smp4yx2f7r0000gn/T/ipykernel_66508/4118419943.py:1: UserWarning: Api key is used with an insecure connection.
  client = QdrantClient(url=QDRANT_URL, api_key="")


uploaded 1000 points to yelp-businesses-qdrant-standalone-openai


In [25]:
def get_embedding(text: str) -> list[float]:
    r = openai_client.embeddings.create(model=EMBED_MODEL, input=[text])
    return r.data[0].embedding


def build_payload_filter(
    *,
    city: str | None = None,
    is_open: bool | None = None,
    min_stars: float | None = None,
    max_price: int | None = None,
    required_categories: list[str] | None = None,
    happy_hour: bool | None = None,
    outdoor_seating: bool | None = None,
    garage_parking: bool | None = None,
) -> Filter | None:
    conds: list[FieldCondition] = []

    if city:
        conds.append(FieldCondition(key="city", match=MatchValue(value=city)))
    if is_open is not None:
        conds.append(FieldCondition(key="is_open", match=MatchValue(value=int(is_open))))
    if min_stars is not None:
        conds.append(FieldCondition(key="stars", range=Range(gte=float(min_stars))))
    if max_price is not None:
        conds.append(FieldCondition(key="attributes_RestaurantsPriceRange2", range=Range(lte=int(max_price))))
    if required_categories:
        conds.append(FieldCondition(key="categories", match=MatchAny(any=required_categories)))
    if happy_hour is not None:
        conds.append(FieldCondition(key="attributes_HappyHour", match=MatchValue(value=bool(happy_hour))))
    if outdoor_seating is not None:
        conds.append(FieldCondition(key="attributes_OutdoorSeating", match=MatchValue(value=bool(outdoor_seating))))
    if garage_parking is not None:
        conds.append(
            FieldCondition(
                key="attributes_BusinessParking_garage",
                match=MatchValue(value=bool(garage_parking)),
            )
        )

    return Filter(must=conds) if conds else None


def retrieve_qdrant(
    query: str,
    k: int = 5,
    *,
    city: str | None = None,
    is_open: bool | None = None,
    min_stars: float | None = None,
    max_price: int | None = None,
    required_categories: list[str] | None = None,
    happy_hour: bool | None = None,
    outdoor_seating: bool | None = None,
    garage_parking: bool | None = None,
):
    """Vector search with optional metadata filters (no Superlinked / NLQ)."""
    qv = get_embedding(query)
    query_filter = build_payload_filter(
        city=city,
        is_open=is_open,
        min_stars=min_stars,
        max_price=max_price,
        required_categories=required_categories,
        happy_hour=happy_hour,
        outdoor_seating=outdoor_seating,
        garage_parking=garage_parking,
    )
    return client.query_points(
        collection_name=COLLECTION_STANDALONE,
        query=qv,
        using=VECTOR_NAME,
        query_filter=query_filter,
        limit=k,
        with_payload=True,
    )


In [26]:
resp = retrieve_qdrant(
    "open cocktail bars New Orleans happy hour outdoor seating garage parking",
    k=5,
    city="New Orleans",
    is_open=True,
    happy_hour=True,
    outdoor_seating=True,
    garage_parking=True,
    required_categories=["Cocktail Bars"],
)
[(p.payload.get("business_id"), p.score, p.payload.get("name"), p.payload.get("city")) for p in resp.points]


[]